# 🧠 JenaXI-1: Jena's own brain, 100% from scratch

This trains **JenaXI-1** (about 110 million parameters, Llama-style). Every weight starts as random numbers. **No other AI model is used.** She learns English from open datasets (FineWeb-Edu, Wikipedia, Project Gutenberg books, TinyStories, plus human-written chats from OpenAssistant and Dolly). Then she learns to be Jena from her own chat lines. It runs on Google's **free** GPU and costs $0.

## What to click (every time)
1. At the top, click **Runtime → Change runtime type → T4 GPU → Save**. Only needed the first time.
2. Click **Runtime → Run all**.
3. A popup asks to connect Google Drive. Click **Connect** and pick your Google account, then **Continue / Allow**. Her progress is saved there.
4. Leave this tab open. You can switch tabs, just don't close it. One session trains for about **3 hours**, then teaches her to talk as Jena (about 15 minutes) and saves **`JenaXI-1.gguf`** to **My Drive → JenaXI**.

## It takes several sessions, and that's normal
The full training is about 1 billion words-pieces, which is about **5 free sessions** (one a day is fine). If Colab disconnects, or you stop it, or you come back tomorrow: **just Run all again**. She continues from her last save. A checkpoint goes to Drive every 15 minutes. Each session also updates `JenaXI-1.gguf`, so she gets smarter every time.

**If it says no GPU is available:** Google's free GPU time ran out for today. Try again later or tomorrow.
You need about 6 GB free on Google Drive.

In [ ]:
#@title 🔌 Step 1: connect Google Drive and check the GPU (approve the Drive popup)
import os, sys, subprocess, signal
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/JenaXI', exist_ok=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No GPU. Click Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all again.")
print("✅ Drive connected. GPU:", gpu)

REPO = '/content/jena-magic-screen'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/pfn000/jena-magic-screen', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'sparse-checkout', 'set', 'colab'], check=True)
print("✅ Jena's training code is ready.")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'pyarrow', 'huggingface_hub'], check=True)
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def run(script, *args):
    """Runs one step, prints its progress live, and stops Run all if the step fails."""
    p = subprocess.Popen([sys.executable, '-u', f'{REPO}/colab/{script}', *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, errors='replace')
    try:
        for line in p.stdout:
            if 'Warning' in line and 'HF_TOKEN' in line: continue
            print(line, end='')
        code = p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT); p.wait()
        raise SystemExit("⏸ Stopped. Progress is saved on your Drive. Run all again any time to continue.")
    if code != 0:
        raise SystemExit(f"❌ {script} stopped with an error (code {code}). Screenshot the red text above and send it to Tab.")

In [ ]:
#@title ⚙️ Step 2: settings (the defaults are good, you can skip this)
import json, os
session_hours = 3.0  #@param {type:"number"}
target_billion_tokens = 1.0  #@param {type:"number"}
RUN = '/content/drive/MyDrive/JenaXI/JenaXI-1'
os.makedirs(RUN, exist_ok=True)
p = f'{RUN}/config.json'
cfg = json.load(open(p)) if os.path.exists(p) else {}
cfg.update(session_hours=float(session_hours), target_tokens=int(target_billion_tokens * 1e9))
json.dump(cfg, open(p, 'w'), indent=1)
print(f"✅ Settings saved: {session_hours} h of pretraining per session, target {target_billion_tokens}B tokens.")

In [ ]:
#@title 🔤 Step 3: Jena's own tokenizer (first session only, about 10 minutes)
run('jenaxi1/train_tokenizer.py')

In [ ]:
#@title 🏋️ Step 4: pretraining from random weights (about 3 hours; resumes where it left off)
# Data downloads and gets prepared in the background while she trains.
run('jenaxi1/pretrain.py')

In [ ]:
#@title 💬 Step 5: teach her to be Jena (persona training, about 15 minutes)
run('jenaxi1/sft.py')

In [ ]:
#@title 📦 Step 6: save JenaXI-1.gguf to your Drive
run('jenaxi1/export.py')

In [ ]:
#@title 🗣️ Step 7: talk to her (sample replies)
run('jenaxi1/sample.py')
import json, os
prog = json.load(open('/content/drive/MyDrive/JenaXI/JenaXI-1/progress.json'))
print("Files are in Google Drive → My Drive → JenaXI:  JenaXI-1.gguf (her brain) and JenaXI-1/ (checkpoints)")
print("Pretraining finished! 🎉" if prog.get('done') else "Pretraining isn't finished yet. Run all again in your next session to keep going.")

## Getting her brain back to Tab
Either:
- **Drive link (easiest):** in Google Drive, open **My Drive → JenaXI**, right-click **JenaXI-1.gguf** → **Share** → under *General access* pick **Anyone with the link** → **Copy link**. Text that link to Tab and Tab will download it.
- **Or download it:** right-click → **Download**, then send the file to Tab.

Tab puts it on Jena's server and the magic-screen site starts using it.